In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
import os, sys, urllib.request, json

from langchain.agents import create_agent
from langchain.tools import tool


@tool
def naver_shop(product: str) -> list[dict]:
    """네이버 쇼핑의 최저가를 검색해서 판매하는곳 링크를 10개 알려준다."""
    client_id = os.getenv("NAVER_CLIENT_ID")
    client_secret = os.getenv("NAVER_CLIENT_SECRET")
    encText = urllib.parse.quote(product)
    url = "https://openapi.naver.com/v1/search/shop?query=" + encText  # JSON 결과
    # url = "https://openapi.naver.com/v1/search/shop.xml?query=" + encText # XML 결과
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()

    shop_result_list = []

    if rescode == 200:
        response_body = response.read()
        result_json = json.loads(response_body.decode("utf-8"))

        for temp in result_json["items"]:
            dict_data = {}
            dict_data["물품명"] = temp["title"]
            dict_data["판매링크"] = temp["link"]
            shop_result_list.append(dict_data)

            # {'물품명':'개미사육장 개미키우기 아크릴  행잉 랙 <b>샤오미</b>온습도계2  1개  1L' , '판매링크':'http://link.coupang.com....'}
    else:
        print("Error Code:" + rescode)

    return shop_result_list


agent = create_agent(
    model="gpt-5.4-mini",
    tools=[naver_shop],
    system_prompt="""너는 네이버쇼핑의 최저가를 알려주는 에이전트야.
    사용자의 질문의 상품명을 가지고 네이버쇼핑의 최저가를 검색해서 판매 물건의 이름과 판매 물건의 주소를 알려줘
    tool결과를 바탕으로만 답변하고 추측해서 답변하지 않도록 주의해줘.""",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "네이버 쇼핑에서 맥북프로m5의 최저가정보를 알려줘",
            }
        ]
    }
)

In [27]:
result

{'messages': [HumanMessage(content='네이버 쇼핑에서 맥북프로m5의 최저가정보를 알려줘', additional_kwargs={}, response_metadata={}, id='4e40ec0f-ed8a-4e6c-bb54-001c58d46908'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 241, 'total_tokens': 263, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DVWV1vASxq1U8h2p6p1CNu3Ih6FAY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9a04-507b-70b2-87d5-9fe7270c835e-0', tool_calls=[{'name': 'naver_shop', 'args': {'product': '맥북프로 m5'}, 'id': 'call_r0R0pVjYlDvUJSbO9VjjV1iH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 241, 'output_toke

In [28]:
result["messages"][-1].content

'네이버 쇼핑에서 확인된 **맥북프로 M5** 최저가 정보입니다.\n\n1. **MDE04KH/A Apple 맥북프로 14 m5 맥북 16GB M5 Pro 블랙**  \n   - 링크: https://smartstore.naver.com/main/products/12879788177\n\n2. **MDE34KH/A Apple 맥북프로 14 m5 맥북 24GB M5 Pro 블랙 1TB**  \n   - 링크: https://smartstore.naver.com/main/products/13289068540\n\n3. **Apple 맥북프로 14 M5 (10C CPU 10C GPU) RAM 16GB SSD 512GB 실버 MDE44KH/A**  \n   - 링크: https://www.lotteon.com/p/product/LO2646352176?sitmNo=LO2646352176_2646352177&ch_no=100065&ch_dtl_no=1000030&entryPoint=pcs&dp_infw_cd=CHT\n\n4. **Apple 맥북 프로 14 M5칩**  \n   - 링크: https://link.coupang.com/re/PCSNAVERPCSDP?pageKey=1092421487&ctag=1092421487&lptag=l031929625681&itemId=60219561938&vendorItemId=93875613388&spec=10305197\n\n5. **Apple 맥북 프로 14 M5칩**  \n   - 링크: https://link.coupang.com/re/PCSNAVERPCSDP?pageKey=1092421487&ctag=1092421487&lptag=l207032327996&itemId=73290237690&vendorItemId=94002473487&spec=10305197\n\n원하시면 제가 **“가장 저렴한 1개만”** 골라서 다시 정리해드릴게요.'